# Analisis estadístico

In [ ]:
%load_ext autoreload
%autoreload 2
from mic_array.patron import MicArray


In [ ]:
ma = MicArray.from_tensor('data/tensores/forte_full_aligned.npz')

In [ ]:
ma.plot(theta = 'ref', dB=True)

## Calibración

In [ ]:
ma.calibrate(
    path="data/media",
    array_pattern="mic_{MIC}_ang_cal.wav",
    ref_pattern="mic_ref_ang_cal.wav"
)

ma.to_spl()   # tensor pasa a Pa


# ma.save("data/tensores/forte_spl.npz")  # guarda en FS, deshace la conversión automáticamente

In [ ]:
ma.plot(theta = 'ref', dB=True)

In [ ]:
from filterbank import FilterBank

# Para más control (fmin/fmax, orden, resoluciones 1/6, 1/12, 1/24) usá FilterBank directo:
# fb = FilterBank(sr=44100, bands='1/3', fmin=20, fmax=20000)
# freqs, levels = fb.leq(signal, method='iir')

# Desde MicArray — guarda los resultados en ma.leq_freqs y ma.leq_levels

# Durante desarrollo — rápido
ma.compute_leq(bands='1/3', method='fft')

# Para el reporte final — IEC 61260 compliant (~2 min)
# ma.compute_leq(bands='1/3', method='iir')

# Acceso a los resultados
# ma.leq_freqs   → [20., 25., 31.5, ..., 20000.]
# ma.leq_levels  → shape (19, 20, 31)  — azimuts × elevaciones × bandas
# ma.leq_bands   → '1/3'

In [ ]:
# Espectro de una posición — barras
ma.plot_leq(azimuth=0, theta='ref', frange=(200, 10000))

# Heatmap: todas las elevaciones de un azimut
ma.plot_leq(azimuth=0)

# Heatmap: todos los azimuts para una elevación
ma.plot_leq(theta=90, frange=(200, 8000), vrange=[60, 94])

# Con rango de color fijo para comparar plots
ma.plot_leq(theta='ref', frange=(200, 8000), vrange=[60, 94])


# Otros colores:
# ma.plot_leq(azimuth=0, colorscale='Turbo')
# ma.plot_leq(theta='ref', colorscale='Jet')
# ma.plot_leq(azimuth=0, theta=90)  # bar chart, colorscale no aplica

In [ ]:
ma.report_leq_global(theta='ref')
ma.report_leq_global(theta=90)

In [ ]:
ma.plot_leq_global(theta='ref', yrange=[70, 100])
ma.plot_leq_global(theta='ref', yrange=[70, 100])

## NOTAS: Detección, ajuste y afinación

In [ ]:
ma.scale = {'Fa4':349.23,'Sol4':392.,'La4':440.,'Sib4':466.16,
            'Do5':523.25,'Re5':587.33,'Mi5':659.25,'Fa5':698.46}

# segments = ma.detect_notes(start_s=0.3)

# Si hay notas que no detecta, bajar confidence
# Más conservador — incluye más frames de transición
segments = ma.detect_notes(start_s=0.3, confidence=0.2, gradient_thresh=50.0)

# Más agresivo — solo la parte más estable de cada nota
# segments = ma.detect_notes(start_s=0.3, confidence=0.4, gradient_thresh=15.0)

In [ ]:
ma.plot_f0(azimuth=0, segments=segments)

In [ ]:
# 3a. Corregir un segmento mal ubicado
ma.edit_segment(segments, azimuth=0, note='La4', start_s=1.86, end_s=2.39)
ma.edit_segment(segments, azimuth=0, note='Sib4', start_s=2.42, end_s=2.96)
ma.edit_segment(segments, azimuth=0, note='Mi5', start_s=4.20, end_s=4.70)
ma.edit_segment(segments, azimuth=0, note='Fa5', start_s=4.78, end_s=5.50)

ma.edit_segment(segments, azimuth=10, note='La4', start_s=1.97, end_s=2.51)
ma.edit_segment(segments, azimuth=10, note='Mi5', start_s=4.50, end_s=4.89)
ma.edit_segment(segments, azimuth=10, note='Fa5', start_s=5, end_s=5.70)

ma.edit_segment(segments, azimuth=40, note='Fa5', start_s=4.80, end_s=5.5)
ma.edit_segment(segments, azimuth=50, note='Fa5', start_s=4.73, end_s=5.5)

ma.edit_segment(segments, azimuth=70, note='Mi5', start_s=4.50, end_s=4.95)

ma.edit_segment(segments, azimuth=90, note='Mi5', start_s=4.95, end_s=5.53)

ma.edit_segment(segments, azimuth=100, note='Sol4', start_s=1.30, end_s=1.85)
ma.edit_segment(segments, azimuth=100, note='Sib4', start_s=2.55, end_s=3.1)
ma.edit_segment(segments, azimuth=100, note='Re5', start_s=3.83, end_s=4.35)
ma.edit_segment(segments, azimuth=100, note='Fa5', start_s=5, end_s=5.80)

ma.edit_segment(segments, azimuth=110, note='Mi5', start_s=4.38, end_s=4.82)
ma.edit_segment(segments, azimuth=110, note='Fa5', start_s=4.90, end_s=5.60)

ma.edit_segment(segments, azimuth=120, note='La4', start_s=1.93, end_s=2.5)
ma.edit_segment(segments, azimuth=120, note='Mi5', start_s=4.38, end_s=4.93)
ma.edit_segment(segments, azimuth=120, note='Fa5', start_s=5, end_s=5.8)

ma.edit_segment(segments, azimuth=130, note='Fa4', start_s=0.62, end_s=1.09)
ma.edit_segment(segments, azimuth=130, note='Re5', start_s=3.59, end_s=4.08)
ma.edit_segment(segments, azimuth=130, note='Mi5', start_s=4.15, end_s=4.63)
ma.edit_segment(segments, azimuth=130, note='Fa5', start_s=4.73, end_s=5.45)


ma.edit_segment(segments, azimuth=150, note='Do5', start_s=3.29, end_s=3.79)
ma.edit_segment(segments, azimuth=150, note='Re5', start_s=3.87, end_s=4.37)
ma.edit_segment(segments, azimuth=150, note='Mi5', start_s=4.49, end_s=5.02)
ma.edit_segment(segments, azimuth=150, note='Fa5', start_s=5.08, end_s=5.88)

ma.edit_segment(segments, azimuth=160, note='La4', start_s=1.95, end_s=2.51)
ma.edit_segment(segments, azimuth=160, note='Re5', start_s=3.85, end_s=4.37)
ma.edit_segment(segments, azimuth=160, note='Mi5', start_s=4.44, end_s=5.05)

ma.edit_segment(segments, azimuth=170, note='La4', start_s=1.97, end_s=2.51)
ma.edit_segment(segments, azimuth=170, note='Mi5', start_s=4.50, end_s=4.89)
ma.edit_segment(segments, azimuth=170, note='Fa5', start_s=5, end_s=5.70)


ma.edit_segment(segments, azimuth=180, note='La4', start_s=1.86, end_s=2.39)
ma.edit_segment(segments, azimuth=180, note='Sib4', start_s=2.42, end_s=2.96)
ma.edit_segment(segments, azimuth=180, note='Mi5', start_s=4.20, end_s=4.70)
ma.edit_segment(segments, azimuth=180, note='Fa5', start_s=4.78, end_s=5.50)


In [ ]:
ma.plot_f0(azimuth=180, segments=segments)

In [ ]:
ma.plot_tune(theta='ref', azimuth=180)

In [ ]:
ma.extract_all_notes(segments)   # ← esto falta

In [ ]:
ma.notes['Mi5'].listen(azimuth=130, theta='ref')

In [ ]:
ma.compute_leq_notes() # Antes de ejecutar esto debe correrse ma.extract_all_notes(segments)

In [ ]:
ma.plot_leq_by_note(theta='ref')